# Challenge — Module 2.2: Sensor Readings Triage

**Format:** one DataFrame, one final report. You must use every concept from 2.2: `DataFrame` construction, `.info()`/`.dtypes`, `.loc`/`.iloc`/boolean masks, `.assign()` for derived columns, and missing-data handling (`.isna()`, `.fillna()`, `.dropna()`). Plus everything from 2.1.

**Scenario.** An IoT pipeline drops a CSV of sensor readings. The data is messy — missing values, wrong dtypes, sentinel values that should be null. You need to clean it and produce a per-sensor summary.

## The problem

The setup cell writes `readings.csv` with columns:

- `sensor_id` (string)
- `reading_ts` (string, ISO format)
- `temperature_c` (float, but with `-999` used as a sentinel for missing)
- `humidity_pct` (float, sometimes blank)
- `battery_v` (float)

### Your task

Load it into a DataFrame and:

1. Replace `-999` in `temperature_c` with proper NaN.
2. Drop any row where `battery_v` is missing (a dead sensor is useless).
3. Fill missing `humidity_pct` with the **column mean** (computed after step 2).
4. Add a derived column `status` using `.assign()`:
   - `'critical'`  if `battery_v < 3.0`
   - `'warning'`   if `3.0 <= battery_v < 3.5`
   - `'ok'`        otherwise
5. Select rows where `temperature_c` is **not null** AND `status != 'ok'` using a boolean mask + `.loc`.
6. From that subset, print:
   - The shape (rows, cols).
   - The `sensor_id` with the highest `temperature_c` (use `.iloc` after sorting, or `.idxmax`).
   - The mean `humidity_pct` of the subset, rounded to 2 decimals.

### Expected output (shape)

```
shape=(3, 6)
hottest_sensor=S02
mean_humidity=52.33
```

(Your numbers should match the data the setup writes — don't hardcode.)

### Restrictions

- Use `.assign()` for the `status` column, not direct assignment (`df['status'] = ...`).
- Use `.loc[mask]` for the final selection, not chained indexing.
- Don't use `pd.to_datetime` yet — that's section 2.3.

### Hints

- `df['temperature_c'].replace(-999, pd.NA)` or `.where(df['temperature_c'] != -999)`.
- For the `status` column inside `.assign`, use `np.select([cond1, cond2], ['critical', 'warning'], default='ok')` or `pd.cut` with bins.
- `df.loc[df['temperature_c'].notna() & (df['status'] != 'ok')]`.

In [ ]:
# --- setup: writes readings.csv ---
import csv
rows = [
    ['sensor_id', 'reading_ts', 'temperature_c', 'humidity_pct', 'battery_v'],
    ['S01', '2026-05-01T08:00:00', '21.4', '55',  '3.7'],
    ['S01', '2026-05-01T09:00:00', '-999', '',    '3.6'],
    ['S02', '2026-05-01T08:00:00', '28.9', '48',  '2.8'],
    ['S02', '2026-05-01T09:00:00', '29.5', '49',  ''   ],
    ['S03', '2026-05-01T08:00:00', '19.0', '',    '3.2'],
    ['S03', '2026-05-01T09:00:00', '-999', '60',  '3.1'],
    ['S04', '2026-05-01T08:00:00', '22.0', '52',  '3.9'],
    ['S05', '2026-05-01T08:00:00', '24.5', '',    '2.9'],
]
with open('readings.csv', 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows(rows)
print('setup done')

setup done


In [ ]:
# your code here
import pandas as pd
import numpy as np

df = pd.read_csv('readings.csv')
df.info()

# 1. -999 -> NaN in temperature_c
# 2. drop rows with missing battery_v
# 3. fill humidity_pct NaNs with the mean
# 4. .assign(status=...) using thresholds on battery_v
# 5. subset via .loc[mask]
# 6. print shape, hottest_sensor, mean_humidity
